# Part 3 — Exercise solutions

⚠️ Run the main `part3_search.ipynb` first — these solutions reuse its embedding
cache (`day1/cache/review_embeddings.npy`).

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../../.env", override=True)
client = genai.Client()
EMBED_MODEL = "gemini-embedding-001"
pd.set_option("display.max_colwidth", 100)

games = pd.read_csv("../../data/games.csv")
df = pd.read_csv("../../data/reviews.csv").merge(games, on="game_id")
doc_vecs = np.load("../cache/review_embeddings.npy")


def embed_query(query):
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=query,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY", output_dimensionality=768),
    )
    q = np.array(result.embeddings[0].values)
    return q / np.linalg.norm(q)

## 5.1 — Scoped search

In [ ]:
def search_game(query, title, top_k=5):
    mask = (df["title"] == title).to_numpy()
    sub_df, sub_vecs = df[mask], doc_vecs[mask]

    scores = sub_vecs @ embed_query(query)
    top = np.argsort(scores)[::-1][:top_k]

    hits = sub_df.iloc[top][["review_id", "recommended", "review_text"]].copy()
    hits.insert(0, "score", scores[top].round(3))
    return hits


search_game("crashes and freezes", "Neon Drift Racers")

## 5.2 — Most similar pair

In [ ]:
S = doc_vecs @ doc_vecs.T
np.fill_diagonal(S, -1)                       # a review is always most similar to itself — exclude
i, j = np.unravel_index(S.argmax(), S.shape)

print(f"similarity {S[i, j]:.3f}\n")
for idx in (i, j):
    print(f"[{df.iloc[idx]['title']}] {df.iloc[idx]['review_text']}\n")

## 5.3 — More like this

In [ ]:
def more_like_this(review_id, top_k=5):
    idx = df.index[df["review_id"] == review_id][0]
    scores = doc_vecs @ doc_vecs[idx]         # the review's vector is already computed
    top = [i for i in np.argsort(scores)[::-1] if i != idx][:top_k]

    hits = df.iloc[top][["review_id", "title", "review_text"]].copy()
    hits.insert(0, "score", scores[top].round(3))
    return hits


more_like_this(df["review_id"].iloc[0])

## 5.4 — Honest search

In [ ]:
def search_honest(query, top_k=5, min_score=0.5):
    scores = doc_vecs @ embed_query(query)
    top = np.argsort(scores)[::-1][:top_k]
    top = top[scores[top] >= min_score]

    if len(top) == 0:
        print(f"No relevant reviews found for {query!r} "
              f"(best score: {scores.max():.3f})")
        return None

    hits = df.iloc[top][["review_id", "title", "review_text"]].copy()
    hits.insert(0, "score", scores[top].round(3))
    return hits


search_honest("quantum physics homework")

In [ ]:
search_honest("pay to win cash shop")

The threshold turns "here are the 5 least-irrelevant results" into an honest
*"nothing matches"*. On Day 3, your agent's retrieval tool must make exactly this
call — an agent that treats garbage matches as evidence will hallucinate with
extra confidence.